# codesearch — сравнение би-энкодеров (GPU)

Прогоняет eval для всех конфигов из `configs/embedder-*.yaml` на одном и том же протоколе
(все 6 языков, 10к записей корпуса — как в прогонах Данила, по 200 запросов на язык,
**без реранкера** — сравниваем чистый retrieval).

Каждая модель выполняется в **отдельном процессе**: если одна упадёт (в т.ч. с CUDA-ошибкой),
остальные досчитаются.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**, затем Runtime → Run all.

Ориентировочное время: ~1–1.5 часа (7 моделей × 10к документов корпуса + 1200 запросов на модель).

In [ ]:
!nvidia-smi

In [ ]:
!rm -rf code-searching-engine
!git clone -b feat/embedder-comparison https://github.com/PopovDanil/code-searching-engine.git
%cd code-searching-engine

In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-cpu tree-sitter tree-sitter-python tree-sitter-java tree-sitter-javascript tree-sitter-go tree-sitter-ruby tree-sitter-php datasets typer pyyaml

In [ ]:
import glob
import os
import re
import subprocess

LANGUAGES = "python,java,javascript,go,ruby,php"
MAX_DATASET_RECORDS = "10000"   # делится между языками поровну (~1666 на язык)
MAX_QUERIES = "200"             # на язык

env = os.environ.copy()
env["PYTHONPATH"] = "src"
env["CUDA_LAUNCH_BLOCKING"] = "1"   # точные CUDA-ошибки вместо отложенных


def parse_results(text):
    """Разбирает блок 'Evaluation Results' из вывода cli evaluate."""
    res, current = {}, None
    tail = text.split("Evaluation Results:")[-1]
    for line in tail.splitlines():
        m_lang = re.match(r"^\s{2}(\S+):\s*$", line)
        m_val = re.match(r"^\s{4}(\S+):\s+([0-9.]+)\s*$", line)
        if m_lang:
            current = m_lang.group(1)
            res[current] = {}
        elif m_val and current is not None:
            res[current][m_val.group(1)] = float(m_val.group(2))
    return res


results = {}
for path in sorted(glob.glob("configs/embedder-*.yaml")):
    name = path.split("embedder-")[1].rsplit(".", 1)[0]
    print("\n" + "=" * 70)
    print(f"  {name}   ({path})")
    print("=" * 70, flush=True)
    proc = subprocess.run(
        [
            "python", "-m", "cli", "evaluate",
            "--languages", LANGUAGES,
            "--max-dataset-records", MAX_DATASET_RECORDS,
            "--max-queries", MAX_QUERIES,
            "--config", path,
        ],
        env=env,
        capture_output=True,
        text=True,
    )
    out = proc.stdout + "\n" + proc.stderr
    parsed = parse_results(out)
    if proc.returncode == 0 and parsed:
        results[name] = parsed
        for lang, vals in parsed.items():
            print(f"  {lang}: " + "  ".join(f"{k}={v:.4f}" for k, v in vals.items()))
    else:
        results[name] = {}
        print(f"  FAILED (exit code {proc.returncode}); хвост лога:")
        print(out[-4000:])

In [ ]:
# Сводка: overall по каждой модели + разбивка по языкам (как в прогонах Данила)
METRICS = ["Recall@1", "Recall@5", "Recall@10", "MRR", "NDCG@10"]
LANGS = ["python", "java", "javascript", "go", "ruby", "php"]

print("### Overall по моделям\n")
print("| Embedder | " + " | ".join(METRICS) + " |")
print("|---" * (len(METRICS) + 1) + "|")
for name, res in results.items():
    overall = res.get("overall", {})
    if overall:
        row = " | ".join(f"{overall.get(m, float('nan')):.4f}" for m in METRICS)
    else:
        row = " | ".join("—" for _ in METRICS)
    print(f"| {name} | {row} |")

print("\n### Разбивка по языкам\n")
for name, res in results.items():
    if not res:
        print(f"\n**{name}** — FAILED")
        continue
    print(f"\n**{name}**\n")
    print("| Language | " + " | ".join(METRICS) + " |")
    print("|---" * (len(METRICS) + 1) + "|")
    for lang in LANGS + ["overall"]:
        vals = res.get(lang)
        if not vals:
            continue
        row = " | ".join(f"{vals.get(m, float('nan')):.4f}" for m in METRICS)
        print(f"| {lang} | {row} |")

## Что дальше

1. Markdown-таблицу из ячейки выше — в `experiments.md` (строка «сравнение би-энкодеров»).
2. Победителя прогнать **с реранкером** (в его yaml поставить `enable_reranking: true`) и на большем корпусе.
3. Если код-специфичная модель ≥ Qwen3-0.6B — меняем дефолт в `example_config.yaml`.